# Bombcell workshop – Open Lab Online, 20 October 2026

**Goal:** go from a Kilosort output folder to a curated set of units, and understand *why* each unit got its label.

1. Run Bombcell
2. Look at the output files
3. Look at units in the GUI
4. Fine-tune the classification

**Data:** one session from the [chronic Neuropixels 2.0 dataset](https://doi.org/10.5522/04/24411841.v2)
(Lebedeva, Okun, Krumin, Carandini; Steinmetz et al. *Science* 2021; CC BY-NC-SA 4.0): mouse visual cortex,
spike-sorted with pyKilosort, while the mouse viewed 112 natural images, 5 repeats each.

## 0. Setup

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # locally, uv has already installed everything
    !pip install -q bombcell

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bombcell as bc

URL = "https://ndownloader.figshare.com/files/61060780"   # AL031_2019-12-13_noRawData.tar (~1.7 GB)
DATA_DIR = Path("/content/data") if IN_COLAB else Path("../data").resolve()  # where the dataset goes
FS = 30000                                                # sampling rate (Hz)

In [ ]:
# Download and extract (~1.7 GB, a few minutes). Skipped if already there.
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not any(DATA_DIR.rglob("spike_times.npy")):
    !curl -sL {URL} | tar -x -C {DATA_DIR}

ks_dir = next(DATA_DIR.rglob("spike_times.npy")).parent
session_dir = ks_dir.parent
save_path = session_dir / "bombcell"
print("kilosort folder:", ks_dir)

## 1. Run Bombcell

Bombcell needs only the Kilosort folder. The raw data (`.ap.bin`) is optional: it improves the
amplitude and SNR metrics, but here we run on templates only.

All settings live in one dictionary, `param`.

In [ ]:
param = bc.get_default_parameters(str(ks_dir), kilosort_version=2)   # pyKilosort ~ Kilosort 2.5
param["computeDrift"] = 1                # estimate per-unit drift
param["computeDistanceMetrics"] = 0      # isolation distance / L-ratio: slower, off by default
param["splitGoodAndMua_NonSomatic"] = 0  # 1 = split non-somatic units into good / MUA

# the classification thresholds
bc.print_qm_thresholds(param) if hasattr(bc, "print_qm_thresholds") else \
    print({k: v for k, v in param.items() if k.startswith(("min", "max"))})

In [ ]:
t0 = time.time()
quality_metrics, param, unit_type, unit_type_string = bc.run_bombcell(str(ks_dir), str(save_path), param)
print(f"done in {time.time() - t0:.0f} s")

**Reading the summary plots** (more in the [wiki](https://github.com/Julie-Fabre/bombcell/wiki/Summary-output-plots)):
- **Waveform overlay:** templates of each class. Do the "noise" ones look like noise?
- **Upset plots:** which metrics exclude which units, and how they overlap.
- **Histograms:** distribution of each metric with its threshold. Is a threshold cutting through the middle of a distribution, or removing almost nothing?

## 2. Output files

In [ ]:
# Bombcell's own output folder
for f in sorted(save_path.rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e3:8.0f} kB  {f.relative_to(save_path)}")

In [ ]:
# ...and one .tsv per metric in the Kilosort folder, so Phy shows them as columns.
# cluster_bc_unitType.tsv holds the labels.
print(sorted(f.name for f in ks_dir.glob("cluster_*.tsv")))
pd.read_csv(ks_dir / "cluster_bc_unitType.tsv", sep="\t").head()

In [ ]:
# One row per unit: label + every metric
qm_table = pd.DataFrame(quality_metrics)
qm_table.insert(0, "bc_label", unit_type_string)
print(qm_table["bc_label"].value_counts())
qm_table.head()

In [ ]:
# Same table, but True/False: does each unit pass each threshold?
bc.make_qm_table(quality_metrics, param, unit_type_string).head()

In [ ]:
# Reload everything later without recomputing
param, quality_metrics, _ = bc.load_bc_results(str(save_path))

In [ ]:
# A methods paragraph for your paper, with the exact parameters you used
bc.print_methods_text(param, quality_metrics)

## 3. Look at units in the GUI

Browse units by class with the buttons. For each unit, ask: does the waveform look neuronal?
Is there a refractory period in the autocorrelogram? Is the amplitude stable over time, or cut off?

You can also label units yourself; Bombcell saves your labels so you can compare with its own.

In [ ]:
try:  # needed for widgets in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

gui = bc.unit_quality_gui(
    ks_dir=str(ks_dir),
    quality_metrics=quality_metrics,
    unit_types=unit_type,
    param=param,
    save_path=str(save_path),
)

✏️ **Exercise:** find one unit you disagree with Bombcell about. Which metric drove its label?
Use `qm_table.loc[qm_table.phy_clusterID == UNIT_ID]` to look at its numbers.

In [ ]:
UNIT_ID = 0  # change me
qm_table.loc[qm_table.phy_clusterID == UNIT_ID].T

## 4. Fine-tune the classification

Thresholds are a choice, not a truth. Relabelling is instant: no need to recompute the metrics.

In [ ]:
def relabel(**new_thresholds):
    p = dict(param, **new_thresholds)
    _, labels = bc.get_quality_unit_type(p, quality_metrics)
    return pd.Series(labels).value_counts()

pd.DataFrame({
    "default": relabel(),
    "strict RPV (5%)": relabel(maxRPVviolations=0.05),
    "lenient RPV (20%)": relabel(maxRPVviolations=0.2),
    "presence > 0.9": relabel(minPresenceRatio=0.9),
    "spatial decay > 0.005": relabel(minSpatialDecaySlopeExp=0.005),
}).fillna(0).astype(int)

✏️ **Exercise:** you want to report single-neuron spike timing (e.g. cross-correlograms between pairs).
Which thresholds would you tighten? Now you want to decode the image from the population: which would you relax?

✏️ **Exercise:** many units here are labelled noise *only* because their spatial decay slope is below `minSpatialDecaySlopeExp`.
This recording uses a Neuropixels 2.0 probe (15 µm between rows). Look at a few of these units in the GUI: are they noise?

In [ ]:
# try your own
relabel(maxRPVviolations=0.1, maxPercSpikesMissing=20, minPresenceRatio=0.7, minNumSpikes=300)